In [1]:
from backtesting import Backtest, Strategy
from backtesting.lib import crossover
import talib, pandas, numpy
import yfinance as yf

C:\Users\DELL\AppData\Local\Programs\Python\Python310\lib\site-packages\backtesting\_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

In [2]:
gc = yf.Ticker("GC=F")
data = gc.history('max')

In [3]:
split_idx = int(len(data)*0.7)

oss_data = data[split_idx:]
iss_data = data[:split_idx]

In [20]:
class EMA_VOL(Strategy):

    fast_period = 10
    slow_period = 30
    vol_period  = 20

    def init(self):
        self.fast_ema = self.I(talib.EMA, self.data.Close, self.fast_period)
        self.slow_ema = self.I(talib.EMA, self.data.Close, self.slow_period)
        self.avg_vol = self.I(talib.EMA, self.data.Volume.astype(float), self.vol_period)

    def next(self):
        price = self.data.Close[-1]
        vol = self.data.Volume[-1]
        if crossover(self.fast_ema, self.slow_ema) and (vol>1.2*self.avg_vol):
            self.buy(sl=0.95*price)
        elif crossover(self.slow_ema, self.fast_ema):
            self.position.close()

In [21]:
bt = Backtest(iss_data,EMA_VOL, cash = 10_000)

In [22]:
stats = bt.optimize(
    fast_period = range(5,25,5),
    slow_period = range(30,55,5),
    vol_period = range(10,50,10),
    maximize = 'Equity Final [$]'
)

Backtest.optimize:   0%|          | 0/80 [00:00<?, ?it/s]

Backtest.run:   0%|          | 0/4505 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/4500 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/4495 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/4505 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/4490 [00:00<?, ?bar/s]

... (72 outputs left out)

Backtest.run:   0%|          | 0/4500 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/4495 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/4485 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/4500 [00:00<?, ?bar/s]

In [23]:
stats

Start                     2000-08-30 00:00...
End                       2018-10-02 00:00...
Duration                   6607 days 00:00:00
Exposure Time [%]                     4.63065
Equity Final [$]                  12780.22479
Equity Peak [$]                   14615.99945
Return [%]                           27.80225
Buy & Hold Return [%]                344.8391
Return (Ann.) [%]                     1.37249
Volatility (Ann.) [%]                 4.54793
CAGR [%]                              0.94005
Sharpe Ratio                          0.30178
Sortino Ratio                         0.42774
Calmar Ratio                          0.07414
Alpha [%]                              5.6143
Beta                                  0.06434
Max. Drawdown [%]                    -18.5131
Avg. Drawdown [%]                    -2.55268
Max. Drawdown Duration     4527 days 00:00:00
Avg. Drawdown Duration      310 days 00:00:00
# Trades                                    4
Win Rate [%]                      

In [24]:
bt.plot()

GridPlot(id='p2695', ...)